In [146]:
import numpy as np
from QuditGates import QuditGates
from QuditCircuit import QuditCircuit
from helper_functions import *

## Initializations

In [147]:
dimension = 4
number_S_qudits = 3 # Number of S qudits in the circuit
number_qudits = 1 + 2*number_S_qudits # Total number of qudits in the circuit 

gates = QuditGates(dimension) # Initialize the QuditGtes class

We will initialize the qudit circuit where we will apply the operators as described by equations (20) and (22) in the paper.

In [148]:
qc = QuditCircuit(number_qudits,dimension)

print("Initial state of the circuit:")
qc.display_state()

Initial state of the circuit:
|0000000> : (1+0j)


We will initialize the qudit circuit where we will apply the operators as described in Section IV of the paper.


In [149]:
qc_gates = QuditCircuit(number_qudits,dimension)

print("Initial state of the gate circuit:")
qc_gates.display_state()

Initial state of the gate circuit:
|0000000> : (1+0j)


## Initialize the circuit

We follow the following structure of the qudits in the circuit

$
    q_0 \mapsto |\psi\rangle_A\\
    q_1 \mapsto |\psi\rangle_{S_1}\\
    q_2 \mapsto |\psi\rangle_{N_1}\\
    \dots\\
    q_{2n} \mapsto |\psi\rangle_{S_{n}}\\
    q_{2n+1} \mapsto |\psi\rangle_{N_{n}}\\
$


We initialize the data qudit $|\psi\rangle_{A}$ in each of the two circuits.

In [150]:
gate_rnd = gates.get_random_unitary()
# gate_rnd = gates.get_I_gate()
qc.apply_gate(gate_rnd, [0]) # Apply a random unitary gate to the first qudit
qc_gates.apply_gate(gate_rnd, [0]) # Apply the same random unitary gate


We initialize all the pairs of $(|\psi\rangle_{S_i},|\psi\rangle_{N_i})$ to the generalized Bell state $\frac{1}{\sqrt{d}}\sum_{p=0}^{d-1}|p\rangle|p\rangle$.

In [151]:
# Initialize the (S, N) pairs in the operator circuit
for i in range(number_S_qudits):
    qc.apply_gate(gates.get_H_gate(), [2*i+1])
    qc.apply_gate(gates.CSUM_gate(), [2*i+1, 2*i+2])

# Initialize the (S, N) pairs in the gate circuit
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate(), [2*i+1])
    qc_gates.apply_gate(gates.CSUM_gate(), [2*i+1, 2*i+2])

Display the quantum state

In [152]:
print("State of the operator circuit after initialization:")
qc.display_state_abs()

print("\n")

print("State of the gate circuit after initialization:")
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")


State of the operator circuit after initialization:
|0000000> : 0.0651+0.0508j => |0000000|=0.0826 => Prob=0.0068
|0000011> : 0.0651+0.0508j => |0000011|=0.0826 => Prob=0.0068
|0000022> : 0.0651+0.0508j => |0000022|=0.0826 => Prob=0.0068
|0000033> : 0.0651+0.0508j => |0000033|=0.0826 => Prob=0.0068
|0001100> : 0.0651+0.0508j => |0001100|=0.0826 => Prob=0.0068
|0001111> : 0.0651+0.0508j => |0001111|=0.0826 => Prob=0.0068
|0001122> : 0.0651+0.0508j => |0001122|=0.0826 => Prob=0.0068
|0001133> : 0.0651+0.0508j => |0001133|=0.0826 => Prob=0.0068
|0002200> : 0.0651+0.0508j => |0002200|=0.0826 => Prob=0.0068
|0002211> : 0.0651+0.0508j => |0002211|=0.0826 => Prob=0.0068
|0002222> : 0.0651+0.0508j => |0002222|=0.0826 => Prob=0.0068
|0002233> : 0.0651+0.0508j => |0002233|=0.0826 => Prob=0.0068
|0003300> : 0.0651+0.0508j => |0003300|=0.0826 => Prob=0.0068
|0003311> : 0.0651+0.0508j => |0003311|=0.0826 => Prob=0.0068
|0003322> : 0.0651+0.0508j => |0003322|=0.0826 => Prob=0.0068
|0003333> : 0.0651

## Construct the encryption operator

### Construct the encryption operator as given by equation (20)

In [153]:
targets = [0]+[2*i+1 for i in range(number_S_qudits)]
enc_matrix = gates.create_Uencryption(number_S_qudits)
qc.apply_gate(enc_matrix, targets=targets)

### Construct the encryption operator as described in section IV

We apply the $P_Z$ operator

In [154]:
# Apply the C(X_d) ladder
qc_gates.apply_gate(gates.CSUM_gate(), targets=[0, 1])
for i in range(number_S_qudits-1):
    qc_gates.apply_gate(gates.CSUM_gate(), targets=[2*i+1, 2*(i+1)+1])

# Apply the Q gate to the last S qudit
qc_gates.apply_gate(gates.Q_gate(), targets=[2*(number_S_qudits-1)+1])

# Apply the inverse of the C(X_d) ladder
for i in range(number_S_qudits-1, 0, -1):
    qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[2*(i-1)+1, 2*(i)+1])
qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[0, 1])

We apply the $P_X$ operator

In [155]:
# Apply the F gate to the qudits A and all the S qudits
qc_gates.apply_gate(gates.get_H_gate(), [0])
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate(), [2*i+1])

# Apply the C(X_d) ladder
qc_gates.apply_gate(gates.CSUM_gate(), targets=[0, 1])
for i in range(number_S_qudits-1):
    qc_gates.apply_gate(gates.CSUM_gate(), targets=[2*i+1, 2*(i+1)+1])

# Apply the Q gate to the last S qudit
qc_gates.apply_gate(gates.Q_gate(), targets=[2*(number_S_qudits-1)+1])

# Apply the inverse of the C(X_d) ladder
for i in range(number_S_qudits-1, 0, -1):
    qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[2*(i-1)+1, 2*(i)+1])
qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[0, 1])

# Apply the F^{dagger} gate to the qudits A and all the S qudits
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate().conj().T, [2*i+1])
qc_gates.apply_gate(gates.get_H_gate().conj().T, [0])

We compare the states obtained for both the circuits, the one where we applied the mathematical operator and the one where we applied the gate construction. We show that for both we obtain the same state, thus confirming the constructions are equivalent.

In [156]:
print("State of the operator circuit after encryption:")
qc.display_state_abs()

print("\n")

print("State of the gate circuit after encryption:")
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")

State of the operator circuit after encryption:
|0000000> : 0.0410-0.0050j => |0000000|=0.0413 => Prob=0.0017
|0000011> : 0.0325+0.0254j => |0000011|=0.0413 => Prob=0.0017
|0000022> : -0.041+0.0050j => |0000022|=0.0413 => Prob=0.0017
|0000033> : 0.0325+0.0254j => |0000033|=0.0413 => Prob=0.0017
|0001100> : 0.0325+0.0254j => |0001100|=0.0413 => Prob=0.0017
|0001111> : -0.041+0.0050j => |0001111|=0.0413 => Prob=0.0017
|0001122> : 0.0325+0.0254j => |0001122|=0.0413 => Prob=0.0017
|0001133> : 0.0410-0.0050j => |0001133|=0.0413 => Prob=0.0017
|0002200> : -0.041+0.0050j => |0002200|=0.0413 => Prob=0.0017
|0002211> : 0.0325+0.0254j => |0002211|=0.0413 => Prob=0.0017
|0002222> : 0.0410-0.0050j => |0002222|=0.0413 => Prob=0.0017
|0002233> : 0.0325+0.0254j => |0002233|=0.0413 => Prob=0.0017
|0003300> : 0.0325+0.0254j => |0003300|=0.0413 => Prob=0.0017
|0003311> : 0.0410-0.0050j => |0003311|=0.0413 => Prob=0.0017
|0003322> : 0.0325+0.0254j => |0003322|=0.0413 => Prob=0.0017
|0003333> : -0.041+0.0

## Construct the decryption operator

In [157]:
choose_pair = 1 # The index of the (S,N) pair that we choose for decryption
SN_target = [2*(choose_pair-1)+1, 2*(choose_pair-1)+2] 
N_targets = []
for i in range(choose_pair-1):
    N_targets.append(2*i+2)
for i in range(choose_pair, number_S_qudits):
    N_targets.append(2*i+2)
N_target = [N_targets[0]]

targets_dec = SN_target + N_targets
print(f"Targets for decryption: {targets_dec}")
print(SN_target)

Targets for decryption: [1, 2, 4, 6]
[1, 2]


### Construct the encryption operator as given by equation (22)

In [158]:
dec_matrix = gates.create_Udecryption1(number_S_qudits)

### Construct the decryption operator as described in section IV

In [159]:
# Apply the T_{S_1, N_1} gate
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)
qc_gates.apply_gate(gates.CSUM_gate().conj().T, SN_target)
qc_gates.apply_gate(gates.get_H_gate().conj().T, [SN_target[0]])
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)

# Apply each of the T_{kl} gate, with k*dim+l in {1, .., dimension^2-1}
for k in range(dimension):
    for l in range(dimension):
        if k == 0 and l == 0:
            continue
    
        # Compute the coefficients
        ck = np.exp(-1j*np.pi*(k)*(k+dimension%2)/dimension)
        cl = np.exp(-1j*np.pi*(l)*(l+dimension%2)/dimension)
        c0 = np.exp(-1j*np.pi*(0)*(0+dimension%2)/dimension)
        c_00 = (c0 * c0)**(-1)
        c_kl = (ck * cl)**(-1)

        control_qudits = SN_target
        control_levels = [k, l]

        gateI = c_kl/c_00 * np.eye(dimension, dtype=np.complex128)
        gateX = np.linalg.matrix_power(gates.get_X_gate(), k)
        gateZ = np.linalg.matrix_power(gates.get_Z_gate(), -l)
        
        # Apply the double-controlled c_kl/c_00 * I_d
        qc_gates.apply_controlled_gate(gateI, control=control_qudits, target=N_target, control_level=control_levels)

        # Apply the double-controlled X^k gate and the double-controlled Z^{-l} gate
        for i in range(1, number_S_qudits):
            target_qudits = [N_targets[i-1]]
            qc_gates.apply_controlled_gate(gateZ, control=control_qudits, target=target_qudits, control_level=control_levels)
            qc_gates.apply_controlled_gate(gateX, control=control_qudits, target=target_qudits, control_level=control_levels)


# Apply the T_{S_1, N_1}^{dagger} gate
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)
qc_gates.apply_gate(gates.get_H_gate(), [SN_target[0]])
qc_gates.apply_gate(gates.CSUM_gate(), SN_target)
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)

qc_gates.apply_gate(gates.C_gate(), targets=SN_target)

# Apply the aditional SWAP gate needed in the circuit
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)


We compare the states obtained for both the circuits, the one where we applied the mathematical operator and the one where we applied the gate construction. We show that for both we obtain the same state, thus confirming the constructions are equivalent.

In [160]:
print("State of the operator circuit after decryption:")
qc.apply_gate(dec_matrix, targets=targets_dec)
qc.display_state_abs()

print("\n")

print("State of the gate circuit after decryption:")
# qc_gates.apply_gate(dec_matrix, targets=targets_dec)
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")

State of the operator circuit after decryption:
|0000000> : 0.0651+0.0508j => |0000000|=0.0826 => Prob=0.0068
|0000011> : 0.0651+0.0508j => |0000011|=0.0826 => Prob=0.0068
|0000022> : 0.0651+0.0508j => |0000022|=0.0826 => Prob=0.0068
|0000033> : 0.0651+0.0508j => |0000033|=0.0826 => Prob=0.0068
|0001100> : 0.0651+0.0508j => |0001100|=0.0826 => Prob=0.0068
|0001111> : 0.0651+0.0508j => |0001111|=0.0826 => Prob=0.0068
|0001122> : 0.0651+0.0508j => |0001122|=0.0826 => Prob=0.0068
|0001133> : 0.0651+0.0508j => |0001133|=0.0826 => Prob=0.0068
|0002200> : 0.0651+0.0508j => |0002200|=0.0826 => Prob=0.0068
|0002211> : 0.0651+0.0508j => |0002211|=0.0826 => Prob=0.0068
|0002222> : 0.0651+0.0508j => |0002222|=0.0826 => Prob=0.0068
|0002233> : 0.0651+0.0508j => |0002233|=0.0826 => Prob=0.0068
|0003300> : 0.0651+0.0508j => |0003300|=0.0826 => Prob=0.0068
|0003311> : 0.0651+0.0508j => |0003311|=0.0826 => Prob=0.0068
|0003322> : 0.0651+0.0508j => |0003322|=0.0826 => Prob=0.0068
|0003333> : 0.0651+0.0

## Test final state

We construct the final state of the circuit, as we have mathematically derived in equation (D8) in paper. We compare this state to the state obtained after applying the encryption and decryption operators to the initial circuit. Thus, we confirm the correctness of the construction.

In [161]:
qudits_to_apply = [i for i in range(1,number_qudits) if i not in SN_target]
qudits_to_apply.append(0)
qudits_to_apply.append(SN_target[1])
qudits_to_apply = np.array(qudits_to_apply).reshape((len(qudits_to_apply)//2, 2))


qc_test = QuditCircuit(number_qudits, dimension)
qc_test.apply_gate(gate_rnd, [SN_target[0]])
for pair in qudits_to_apply:
    qc_test.apply_gate(gates.get_H_gate(), [pair[0]])
    qc_test.apply_gate(gates.CSUM_gate(), targets=[pair[0], pair[1]])


In [162]:
print(compare_quantum_states(qc, qc_test))
print(compare_quantum_states(qc_gates, qc_test))

True
True
